# 00. 공통 엔진 (Trainer / Registry / Config / TaskAdapter)

이 노트북은 `cv_boilerplate`의 **task-agnostic 공통 엔진**이 어떻게 동작하는지를 다룬다.
특정 태스크의 데이터 성질에는 들어가지 않는다 (그 내용은 `toy/01_toy_cls.ipynb` 이후에서 다룬다).

다루는 내용:

1. Config 로더 — YAML 상속, `--set` override, 검증 실패 동작, `config.resolved.yaml`의 의미
2. Registry — 모델·데이터셋·transform·metric·adapter가 이름으로 등록·조회되는 방식
3. TaskAdapter 계약 — 엔진이 어댑터에 무엇을 요구하고, 어댑터가 태스크 차이를 어디서 흡수하는가
4. Trainer 루프 — epoch 진행, `model.train()`/`model.eval()` 경계, `no_grad()` 범위, metric 리셋
   시점, checkpoint 저장, monitor 기반 best 선택
5. 비표준 학습 훅 — `on_fit_start` / `on_fit_end`가 왜 필요하고 어디서 호출되는가
6. 엔진 순수성 — 공통 루프에 태스크 이름 분기가 없다는 사실을 코드로 확인

`toy_cls` 설정(`configs/toy/toy_cls.yaml`)을 예제로 쓰되, 여기서 확인하는 사실은 4개 태스크
전부에 동일하게 적용된다. 로컬 데이터셋이나 체크포인트 없이, CPU에서 완주한다.

In [1]:
import os
import sys


def find_repo_root(start):
    """Walk up from the notebook's own directory to the repository root (identified by the
    presence of `src/`), so `import src...` resolves regardless of where this notebook is run
    from (nbconvert executes with cwd set to the notebook's own directory)."""
    path = os.path.abspath(start)
    while not os.path.isdir(os.path.join(path, "src")):
        parent = os.path.dirname(path)
        if parent == path:
            raise RuntimeError("could not locate repository root (no 'src/' found above cwd)")
        path = parent
    return path


REPO_ROOT = find_repo_root(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# Mirrors src/__main__.py: arm the offline guard before any module that might touch the
# network at import time, then import src.tasks so every @DATASETS/@MODELS/@ADAPTERS/...
# decorator across the 4 tasks (+ toy) actually runs and populates the registries. Until this
# import happens, every Registry in src.core.registry is empty.
from src.core.offline import enable_offline_guard

enable_offline_guard()
import src.tasks  # noqa: F401,E402

print("cwd:", os.getcwd())

import inspect
import torch

cwd: /mnt/d/projects/nampluskr/00_review/260818_cv-boilerplate


## 1. Config 로더

설정은 YAML 파일이며 `_base`로 다른 YAML을 상속할 수 있다. `src/core/config.py`의
`load_and_merge_base`가 재귀적으로 병합한다. `configs/toy/toy_cls.yaml`은 `_base`가 없는 단독
config라서, 상속 자체를 보여주는 예로는 실제로 `_base: _base.yaml`을 쓰는
`configs/classification/resnet50.yaml`을 먼저 본다 (YAML을 파싱만 할 뿐 데이터셋·체크포인트를
읽지 않으므로 오프라인에서도 안전하다).

In [2]:
from src.core.config import load_and_merge_base, load_raw_yaml

raw = load_raw_yaml("configs/classification/resnet50.yaml")
print("raw file (before _base merge):")
print(sorted(raw.keys()))
print("_base:", raw.get("_base"))

raw file (before _base merge):
['_base', 'model']
_base: _base.yaml


In [3]:
merged_resnet50 = load_and_merge_base("configs/classification/resnet50.yaml")
base_only = load_raw_yaml("configs/classification/_base.yaml")

print("resnet50.yaml (raw) top-level keys:      ", sorted(raw.keys()))
print("_base.yaml top-level keys:                ", sorted(base_only.keys()))
print("merged top-level keys:                    ", sorted(merged_resnet50.keys()))
print("resnet50.yaml에만 있던 키 (merge 후에도 유지):", sorted(set(raw.keys()) - set(base_only.keys())))
print("_base.yaml에서 새로 합류한 키:              ",
      sorted(set(merged_resnet50.keys()) - set(raw.keys())))

resnet50.yaml (raw) top-level keys:       ['_base', 'model']
_base.yaml top-level keys:                 ['adapter', 'data', 'loss', 'meta', 'metrics', 'optim', 'output', 'runtime', 'train']
merged top-level keys:                     ['adapter', 'data', 'loss', 'meta', 'metrics', 'model', 'optim', 'output', 'runtime', 'train']
resnet50.yaml에만 있던 키 (merge 후에도 유지): ['_base', 'model']
_base.yaml에서 새로 합류한 키:               ['adapter', 'data', 'loss', 'meta', 'metrics', 'optim', 'output', 'runtime', 'train']


`_base`는 리스트도 허용하고, 병합은 딕셔너리를 재귀적으로 합치는 deep-merge다 (리스트는 override가
통째로 대체). `resnet50.yaml`에는 없던 `runtime`/`train`/`output` 같은 최상위 키가 `_base.yaml`
에서 합류하는 것을 확인했다.

이 노트북의 나머지는 다시 `toy_cls.yaml`을 쓴다 -- 상속 여부와 무관하게, 아래에서 다루는
`--set` override·검증·`config.resolved.yaml`은 모든 config에 동일하게 적용된다.

In [4]:
merged = load_and_merge_base("configs/toy/toy_cls.yaml")
print("merged top-level keys:", sorted(merged.keys()))
print()
print("merged['runtime']:", merged["runtime"])
print("merged['train']:", merged["train"])

merged top-level keys: ['adapter', 'data', 'loss', 'meta', 'metrics', 'model', 'optim', 'output', 'runtime', 'train']

merged['runtime']: {'seed': 42, 'device': 'cpu', 'amp': False, 'deterministic': 'strict', 'allow_network': False}
merged['train']: {'epochs': 5, 'grad_clip': None, 'monitor': {'metric': 'accuracy', 'mode': 'max'}, 'log_interval': 10, 'save_last': True}


### `--set` override

CLI의 `--set data.batch_size=4` 같은 override는 `apply_overrides` -> `apply_set_override`가
점(`.`) 구분 경로를 따라 내려가며 리프 값을 바꾼다. 존재하지 않는 키는 에러다 (조용히 새 키를 만들지
않는다).

In [5]:
from src.core.config import apply_overrides
from src.core.errors import ConfigError

overridden = apply_overrides(merged, ["data.batch_size=4", "train.epochs=1"])
print("data.batch_size:", overridden["data"]["batch_size"])
print("train.epochs:", overridden["train"]["epochs"])

try:
    apply_overrides(merged, ["data.no_such_key=1"])
except ConfigError as e:
    print("\nConfigError (as expected):", e)

data.batch_size: 4
train.epochs: 1

ConfigError (as expected): --set key 'data.no_such_key' references nonexistent key 'no_such_key'.


### 검증 실패 동작

`validate_config`는 필수 top-level 키, 타입, registry 등록 여부, 경로 존재, monitor metric이
`metrics`에 선언되어 있는지 등을 확인한다. 하나라도 어긋나면 학습을 시작하지 않고 `ConfigError`로
즉시 실패한다 (조용히 기본값으로 대체하지 않는다).

In [6]:
from src.core.config import validate_config

# 1) 정상 케이스
ok_config = validate_config(load_and_merge_base("configs/toy/toy_cls.yaml"))
print("validate_config ok:", ok_config["meta"]["task_name"])

# 2) monitor.metric이 metrics 목록에 없는 경우
import copy
bad_config = copy.deepcopy(ok_config)
bad_config["train"]["monitor"]["metric"] = "not_a_real_metric"
try:
    validate_config(bad_config)
except ConfigError as e:
    print("\nConfigError (monitor metric not declared):", e)

# 3) registry에 없는 모델 이름
bad_config2 = copy.deepcopy(ok_config)
bad_config2["model"]["name"] = "not_a_real_model"
try:
    validate_config(bad_config2)
except ConfigError as e:
    print("\nConfigError (unregistered model):", e)

validate_config ok: toy_cls

ConfigError (monitor metric not declared): train.monitor.metric 'not_a_real_metric' is not in metrics ['accuracy'].

ConfigError (unregistered model): model.name 'not_a_real_model' is not registered in namespace 'model'. Available: ['custom_ae_anomaly', 'custom_cnn_cls', 'custom_fcos_det', 'custom_unet_seg', 'deeplabv3_resnet50_seg', 'efficientad_anomaly', 'efficientnet_b0_cls', 'fasterrcnn_r50_fpn_det', 'fcn_resnet50_seg', 'resnet50_cls', 'stfpm_anomaly', 'toy_ae', 'toy_ae_trainstep', 'toy_cnn', 'toy_cnn_small', 'toy_det_head', 'toy_mlp', 'toy_segnet', 'yolov8n_det']


### `config.resolved.yaml`의 의미

`train`/`evaluate`/`benchmark` 커맨드는 실행 시작 시 `_base` 병합과 `--set` override가 모두 끝난
**최종 config**를 `run_dir/config.resolved.yaml`에 저장한다 (`src/cli/commands.py`의 `train()`,
`evaluate()`, `src/bench/runner.py`의 벤치마크 split 실행 참고). `predict`는 이 파일을 남기지
않는다 -- 체크포인트에 이미 학습 당시 config가 저장되어 있고, `predict`는 새로 학습하지 않으므로
공정 비교 대상이 아니기 때문이다. 원본 YAML 여러 개와 CLI override를 다시 추적하지 않고 이 파일
하나만 보면 그 실행이 정확히 어떤 설정으로 돌았는지 알 수 있다 — 이것이 공정 비교(동일 해상도·
augmentation·optimizer·seed)를 사후에 검증하는 근거 자료다.

In [7]:
import glob

sample_resolved = sorted(glob.glob("outputs/benchmarks/cls_baseline/splits/*/config.resolved.yaml"))
if sample_resolved:
    print(f"예시: {sample_resolved[0]}")
    with open(sample_resolved[0], encoding="utf-8") as f:
        print(f.read()[:600], "...")
else:
    print("outputs/benchmarks/cls_baseline가 아직 없다 (v0.1 벤치마크를 먼저 실행해야 한다).")
    print("이 노트북 자체는 이 파일 없이도 계속 진행할 수 있다 -- config.resolved.yaml은")
    print("실행 시 자동으로 생성되는 산출물이다.")

예시: outputs/benchmarks/cls_baseline/splits/custom_cnn/config.resolved.yaml
meta:
  task_name: classification
  description: oxford_pets 37-breed classification baseline
runtime:
  seed: 42
  device: cuda
  amp: false
  deterministic: warn
  allow_network: false
data:
  name: oxford_pets_cls
  root: /mnt/d/datasets/oxford_pets
  params: {}
  image_size:
  - 224
  - 224
  batch_size: 32
  num_workers: 4
  drop_last: false
  split:
    mode: file
    path: configs/splits/oxford_pets_subset_cls.json
  transform:
    train:
      name: cls_train
      params: {}
    eval:
      name: cls_eval
      params: {}
loss:
  name: cross_entropy
  params:
    label_smoothing: 0.0
 ...


## 2. Registry

`src/core/registry.py`의 `Registry`는 이름(string) -> 클래스/함수를 매핑하는 단순한 딕셔너리
래퍼다. 각 태스크 모듈은 `@DATASETS.register("이름")`, `@MODELS.register("이름")` 같은 데코레이터로
자신을 등록한다. 엔진과 CLI는 이 이름을 config에서 읽어 `Registry.build(name, *args, **params)`로
인스턴스를 만들 뿐, 어떤 태스크의 무엇을 만드는지 알지 못한다.

이 등록은 **모듈을 import하는 시점**에 데코레이터가 실행되며 일어난다. 1장의 `validate_config`가
`data.name`을 이미 인식할 수 있었던 것은, 맨 첫 셀의 `import src.tasks`(`src/__main__.py`와 동일한
지점)가 `anomaly`/`classification`/`detection`/`segmentation`/`toy` 5개 패키지를 모두 import해
그 안의 모든 `@DATASETS.register(...)` 등을 실행시켰기 때문이다. `src.core.registry`만 import하고
`src.tasks`를 import하지 않으면 아래 registry들은 전부 비어 있다.

In [8]:
from src.core.registry import ADAPTERS, DATASETS, LOSSES, METRICS, MODELS, TRANSFORMS

# 각 registry에 등록된 전체 이름 (4개 태스크 + toy 전부 포함)
for registry in (DATASETS, MODELS, ADAPTERS, TRANSFORMS, METRICS):
    print(f"{registry.namespace:<10} ({len(registry.keys())}):", registry.keys())

dataset    (8): ['mvtec_anomaly', 'oxford_pets_cls', 'oxford_pets_det', 'oxford_pets_seg', 'toy_anomaly', 'toy_cls', 'toy_det', 'toy_seg']
model      (19): ['custom_ae_anomaly', 'custom_cnn_cls', 'custom_fcos_det', 'custom_unet_seg', 'deeplabv3_resnet50_seg', 'efficientad_anomaly', 'efficientnet_b0_cls', 'fasterrcnn_r50_fpn_det', 'fcn_resnet50_seg', 'resnet50_cls', 'stfpm_anomaly', 'toy_ae', 'toy_ae_trainstep', 'toy_cnn', 'toy_cnn_small', 'toy_det_head', 'toy_mlp', 'toy_segnet', 'yolov8n_det']
adapter    (8): ['anomaly', 'classification', 'detection', 'segmentation', 'toy_anomaly', 'toy_cls', 'toy_det', 'toy_seg']
transform  (9): ['anomaly_default', 'cls_eval', 'cls_train', 'det_eval', 'det_train', 'seg_eval', 'seg_train', 'toy_eval', 'toy_train']
metric     (9): ['accuracy', 'dice', 'image_auroc', 'macro_f1', 'map', 'map_50_95', 'miou', 'pixel_auroc', 'top1_accuracy']


In [9]:
print(inspect.getsource(type(MODELS).register))
print(inspect.getsource(type(MODELS).build))

    def register(self, name):
        def decorator(cls_or_fn):
            if name in self.entries:
                raise RegistryError(
                    f"'{name}' is already registered in namespace '{self.namespace}'."
                )
            self.entries[name] = cls_or_fn
            return cls_or_fn

        return decorator

    def build(self, name, *args, **params):
        target = self.get(name)
        return target(*args, **params)



In [10]:
from src.core.errors import RegistryError

try:
    MODELS.build("not_a_real_model")
except RegistryError as e:
    print("RegistryError (as expected):", e)

RegistryError (as expected): 'not_a_real_model' is not registered in namespace 'model'. Available: ['custom_ae_anomaly', 'custom_cnn_cls', 'custom_fcos_det', 'custom_unet_seg', 'deeplabv3_resnet50_seg', 'efficientad_anomaly', 'efficientnet_b0_cls', 'fasterrcnn_r50_fpn_det', 'fcn_resnet50_seg', 'resnet50_cls', 'stfpm_anomaly', 'toy_ae', 'toy_ae_trainstep', 'toy_cnn', 'toy_cnn_small', 'toy_det_head', 'toy_mlp', 'toy_segnet', 'yolov8n_det']


`MODELS.build("toy_cnn", num_classes=4)`처럼, config의 `model.name`과 `model.params`가 그대로
`Registry.build`의 인자가 된다. 이 노트북이 나중에 여러 모델을 이름만 바꿔가며 로드할 수 있는 이유가
이것이다 -- 엔진 쪽 코드는 전혀 바뀌지 않는다.

In [11]:
toy_model = MODELS.build("toy_cnn", num_classes=4)
print(type(toy_model).__name__, "params:", sum(p.numel() for p in toy_model.parameters()))

ToyCnnClassifier params: 23844


## 3. TaskAdapter 계약

`src/core/adapter.py`의 `TaskAdapter`는 엔진과 태스크 사이의 경계다. `Trainer`(4장)는 이 7개
추상 메서드와 `batch_size()`만 호출하며, 그 안에서 배치를 모델에 어떻게 넣고 loss/metric을 어떻게
만드는지는 전적으로 어댑터(태스크)가 책임진다.

In [12]:
print(inspect.getsource(__import__("src.core.adapter", fromlist=["TaskAdapter"]).TaskAdapter))

class TaskAdapter(ABC):
    """Encapsulates how one batch is forwarded, how loss and predictions are produced,
    and how metrics are updated. The engine knows nothing else about a task."""

    def __init__(self, loss_fn, metrics, **params):
        self.loss_fn = loss_fn
        self.metrics = metrics

    # --- required ---
    @abstractmethod
    def train_step(self, model, batch, device) -> dict:
        """Return {"loss": scalar Tensor with grad, "loss_dict": {str: float}}."""

    @abstractmethod
    def eval_step(self, model, batch, device) -> dict:
        """Return {"loss": scalar Tensor or None, "outputs": Any}."""

    @abstractmethod
    def update_metrics(self, outputs) -> None:
        ...

    @abstractmethod
    def compute_metrics(self) -> dict:
        ...

    @abstractmethod
    def reset_metrics(self) -> None:
        ...

    @abstractmethod
    def predict_step(self, model, batch, device) -> list:
        """Return one serializable prediction per sample in the 

`toy_cls` 어댑터가 이 계약을 어떻게 구현하는지 보자. Classification은 4개 태스크 중 가장 단순한
경우다 -- `train_step`이 이미지를 모델에 통과시키고 loss를 계산해 돌려줄 뿐이다.

In [13]:
from src.core.registry import ADAPTERS

ToyClsAdapter = ADAPTERS.get("toy_cls")
print(inspect.getsource(ToyClsAdapter))

@ADAPTERS.register("toy_cls")
class ToyClsAdapter(TaskAdapter):
    def train_step(self, model, batch, device):
        images, targets = batch
        images, targets = images.to(device), targets.to(device)
        return run_model_step(model, images, targets, self.loss_fn)

    def eval_step(self, model, batch, device):
        images, targets = batch
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss = self.loss_fn(outputs, targets)
        preds = torch.argmax(outputs, dim=1)
        return {"loss": loss, "outputs": {"preds": preds.cpu(), "targets": targets.cpu()}}

    def update_metrics(self, outputs):
        preds, targets = outputs["outputs"]["preds"], outputs["outputs"]["targets"]
        for metric in self.metrics.values():
            metric.update(preds, targets)

    def compute_metrics(self):
        return {name: float(metric.compute()) for name, metric in self.metrics.items()}

    def reset_metrics(self):
     

`Trainer`는 `train_step`이 반환하는 dict의 `"loss"`만 보고 `backward()`를 호출한다. 어댑터가
내부적으로 이미지를 어떻게 device로 옮기고, 어떤 loss 함수를 쓰고, 몇 개의 metric을 업데이트하는지는
`Trainer` 입장에서 완전히 불투명하다 -- 이것이 태스크 차이를 흡수하는 지점이다.

## 4. Trainer 루프

`src/core/engine.py`의 `Trainer`가 실제 학습을 진행한다. 먼저 `toy_cls`를 처음부터 끝까지 학습시켜
보면서, 각 단계가 `src/cli/commands.py`의 `train()`이 호출하는 것과 동일한 함수들로 구성됨을
확인한다 -- 노트북은 이 파이프라인을 재구현하지 않고 그대로 호출한다.

In [14]:
from src.cli.commands import bind_class_names, build_components, build_dataset, build_transforms
from src.core.builders import build_dataloader, build_optimizer, build_scheduler
from src.core.config import resolve_config
from src.core.context import RunContext
from src.core.engine import Trainer
from src.core.logger import MetricsCsvWriter, setup_logger

config = resolve_config("configs/toy/toy_cls.yaml")
config = validate_config(config)

run_dir = os.path.join("outputs", "notebooks", "00_engine_toy_cls")
os.makedirs(run_dir, exist_ok=True)

ctx = RunContext(config, run_dir, allow_test_split=False)
ctx.start("notebooks/toy/00_engine.ipynb")
ctx.setup_seed()

print("device:", ctx.device, " epochs:", ctx.epochs, " seed:", ctx.seed)

device: cpu  epochs: 5  seed: 42


In [15]:
transform_train, transform_eval = build_transforms(config)
model, adapter = build_components(config)
adapter.to(ctx.device)

train_ds = build_dataset(config, "train", transform_train)
valid_ds = build_dataset(config, "valid", transform_eval)
bind_class_names(adapter, train_ds)

print("model:", type(model).__name__)
print("adapter:", type(adapter).__name__)
print("train samples:", len(train_ds), " valid samples:", len(valid_ds))

model: ToyCnnClassifier
adapter: ToyClsAdapter
train samples: 64  valid samples: 32


In [16]:
train_loader = build_dataloader(train_ds, config["data"], "train", adapter, ctx.seed,
                                 config["runtime"]["device"])
valid_loader = build_dataloader(valid_ds, config["data"], "valid", adapter, ctx.seed,
                                 config["runtime"]["device"])

optimizer = build_optimizer(config["optim"], model)
scheduler = build_scheduler(config["optim"], optimizer)
print("optimizer:", type(optimizer).__name__, " scheduler:", type(scheduler).__name__)

optimizer: AdamW  scheduler: CosineAnnealingLR


In [17]:
logger = setup_logger(run_dir, "INFO")
metrics_writer = MetricsCsvWriter(run_dir, [m["name"] for m in config["metrics"]])
checkpoint_dir = os.path.join(run_dir, "checkpoints")
trainer = Trainer(logger=logger, metrics_writer=metrics_writer, checkpoint_dir=checkpoint_dir)

best = trainer.fit(model, adapter, train_loader, valid_loader, optimizer, scheduler, ctx)
ctx.finish()
print("\nbest valid", config["train"]["monitor"]["metric"], "=", best)

[2026-08-19 19:37:55] [INFO] epoch 1 train done in 0.1s loss={'loss': 1.411376729607582}


[2026-08-19 19:37:55] [INFO] valid epoch=1 loss=1.4161 metrics={'accuracy': 0.21875} in 0.0s


[2026-08-19 19:37:57] [INFO] epoch 2 train done in 0.0s loss={'loss': 1.3781511634588242}


[2026-08-19 19:37:57] [INFO] valid epoch=2 loss=1.4332 metrics={'accuracy': 0.15625} in 0.0s


[2026-08-19 19:37:57] [INFO] epoch 3 train done in 0.0s loss={'loss': 1.3781891614198685}


...

[2026-08-19 19:37:58] [INFO] epoch 4 train done in 0.0s loss={'loss': 1.373585656285286}


[2026-08-19 19:37:58] [INFO] valid epoch=4 loss=1.4389 metrics={'accuracy': 0.15625} in 0.0s


[2026-08-19 19:37:59] [INFO] epoch 5 train done in 0.0s loss={'loss': 1.370521068572998}


[2026-08-19 19:37:59] [INFO] valid epoch=5 loss=1.4382 metrics={'accuracy': 0.15625} in 0.0s



best valid accuracy = 0.21875


### `train()`/`eval()` 경계, `no_grad()`, metric 리셋

`Trainer.fit`은 매 epoch마다 `_train_epoch`(학습)과 `evaluate`(검증)를 번갈아 부른다. 아래는
`Trainer` 소스 그대로다. 확인할 지점:

- `_train_epoch`은 맨 앞에서 `model.train()`을 부른다.
- `evaluate`는 맨 앞에서 `model.eval()`을 부르고, 전체 루프를 `with torch.no_grad():`로 감싼다.
- `evaluate`는 시작할 때 `adapter.reset_metrics()`를 불러 이전 split/epoch의 누적값이 섞이지 않게
  한다.
- best 모델 저장은 `train.monitor`의 `metric`/`mode`(max/min)를 기준으로 하며, epoch마다 비교해
  개선됐을 때만 `checkpoints/best.pth`를 갱신한다. `save_last`가 켜져 있으면 매 epoch `last.pth`도
  갱신한다.

In [18]:
print(inspect.getsource(Trainer.fit))

    def fit(self, model, adapter, train_loader, valid_loader, optimizer, scheduler, ctx,
            start_epoch=1, best_metric=None):
        model.to(ctx.device)
        scaler = torch.amp.GradScaler("cuda", enabled=(ctx.amp and ctx.device.type == "cuda"))
        loaders = {"train": train_loader, "valid": valid_loader}
        monitor_metric = ctx.config["train"]["monitor"]["metric"]
        monitor_mode = ctx.config["train"]["monitor"]["mode"]

        adapter.on_fit_start(model, loaders, ctx.device)
        best_checkpoint_path = os.path.join(self.checkpoint_dir, "best.pth") if self.checkpoint_dir else None

        for epoch in range(start_epoch, ctx.epochs + 1):
            adapter.on_epoch_start(model, epoch)
            self._train_epoch(model, adapter, train_loader, optimizer, scaler, ctx, epoch)
            if scheduler is not None:
                scheduler.step()
            valid_results = self.evaluate(model, adapter, valid_loader, ctx, epoch=epoch, split="valid")
      

In [19]:
print(inspect.getsource(Trainer._train_epoch))
print(inspect.getsource(Trainer.evaluate))

    def _train_epoch(self, model, adapter, loader, optimizer, scaler, ctx, epoch):
        model.train()
        start = time.perf_counter()
        total_count = 0
        loss_scalar_sum = 0.0
        loss_dict_sum = {}
        log_interval = ctx.config["train"]["log_interval"]

        for step, batch in enumerate(loader, start=1):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(ctx.amp and ctx.device.type == "cuda")):
                step_out = adapter.train_step(model, batch, ctx.device)
            loss = step_out["loss"]
            scaler.scale(loss).backward()
            if ctx.grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), ctx.grad_clip)
            scaler.step(optimizer)
            scaler.update()

            batch_count = adapter.batch_size(batch)
            total_count += batch_count
            loss_scalar_sum += float(loss.detac

In [20]:
import pandas as pd

epoch_log = pd.read_csv(os.path.join(run_dir, "metrics_epoch.csv"))
epoch_log

,epoch,split,loss,accuracy,lr,elapsed_sec
0,1,train,1.411377,NaN,0.001000,0.141386
1,1,valid,1.416086,0.21875,0.000000,0.012864
2,2,train,1.378151,NaN,0.000905,0.037470
3,2,valid,1.433152,0.15625,0.000000,0.010383
4,3,train,1.378189,NaN,0.000658,0.043277
5,3,valid,1.451978,0.15625,0.000000,0.009594
6,4,train,1.373586,NaN,0.000352,0.039387
7,4,valid,1.438875,0.15625,0.000000,0.009103
8,5,train,1.370521,NaN,0.000105,0.039444
9,5,valid,1.438246,0.15625,0.000000,0.008359


`fit()`이 끝나면 best 체크포인트를 다시 로드한다 -- 마지막 epoch이 아니라 monitor 기준 최고 epoch의
가중치가 최종 모델이 되도록 하기 위해서다. `checkpoints/best.pth`가 실제로 저장됐는지 확인한다.

In [21]:
best_path = os.path.join(checkpoint_dir, "best.pth")
last_path = os.path.join(checkpoint_dir, "last.pth")
print("best.pth exists:", os.path.isfile(best_path))
print("last.pth exists:", os.path.isfile(last_path))

checkpoint = torch.load(best_path, map_location="cpu", weights_only=False) if os.path.isfile(best_path) else None
if checkpoint is not None:
    print("best.pth epoch:", checkpoint["epoch"], " best_metric:", checkpoint["best_metric"])

best.pth exists: True
last.pth exists: True
best.pth epoch: 1  best_metric: 0.21875


## 5. 비표준 학습 훅: `on_fit_start` / `on_fit_end`

`Trainer.fit`은 학습 루프 앞뒤로 `adapter.on_fit_start(...)`와 `adapter.on_fit_end(...)`를 부른다
(둘 다 `TaskAdapter`의 기본 구현은 no-op). 이 훅이 필요한 이유는, epoch 단위 `train_step`/`eval_step`
만으로는 표현할 수 없는 계산이 있기 때문이다.

대표 사례가 `toy_anomaly` 어댑터다. Anomaly Detection은 "정상/비정상" 라벨이 아니라 재구성 오차
점수로 이상을 판정하므로, **valid 셋 전체(정상과 이상 이미지가 섞여 있다)를 학습이 끝난 뒤 한 번 더
통과시켜 재구성 오차의 평균과 표준편차로 threshold를 계산**해야 한다. `valid` split은 절반이
인위적으로 결함을 주입한 이미지다 (04_toy_anomaly.ipynb에서 다시 확인한다) -- "정상 데이터만으로
학습한다"는 것은 **train** split에만 해당하는 얘기이고, threshold 계산에 쓰는 **valid** split은
정상/이상이 섞여 있어도 된다 (그 반응 차이를 이용해 threshold를 정하는 것이므로). 이 계산은 어느 한
epoch에 속하지 않으므로 `on_fit_end`에만 자연스럽게 들어간다.

In [22]:
from src.tasks.toy.adapter import ToyAnomalyAdapter

print(inspect.getsource(ToyAnomalyAdapter.on_fit_end))

    def on_fit_end(self, model, loaders, device):
        model.eval()
        scores = []
        with torch.no_grad():
            for images, targets in loaders["valid"]:
                images = images.to(device)
                reconstruction = model(images)
                error_map = (reconstruction - images).pow(2).mean(dim=1)
                scores.append(error_map.flatten(1).amax(dim=1).cpu())
        if scores:
            all_scores = torch.cat(scores)
            self.threshold = float(all_scores.mean() + 2 * all_scores.std())



`fit()` 소스를 다시 보면, `on_fit_end`는 **best 체크포인트를 재로드한 이후**에 호출된다. 훅이
학습 중간의 임의 epoch이 아니라 최종 선택된 모델을 기준으로 threshold 같은 보정값을 계산하도록 하기
위한 순서다. 실제로 `toy_anomaly`를 학습시켜 threshold가 채워지는 것을 확인한다.

In [23]:
anomaly_config = validate_config(resolve_config("configs/toy/toy_anomaly.yaml"))
anomaly_run_dir = os.path.join("outputs", "notebooks", "00_engine_toy_anomaly")
os.makedirs(anomaly_run_dir, exist_ok=True)

anomaly_ctx = RunContext(anomaly_config, anomaly_run_dir, allow_test_split=False)
anomaly_ctx.start("notebooks/toy/00_engine.ipynb")
anomaly_ctx.setup_seed()

a_transform_train, a_transform_eval = build_transforms(anomaly_config)
a_model, a_adapter = build_components(anomaly_config)
a_adapter.to(anomaly_ctx.device)
a_train_ds = build_dataset(anomaly_config, "train", a_transform_train)
a_valid_ds = build_dataset(anomaly_config, "valid", a_transform_eval)
bind_class_names(a_adapter, a_train_ds)

a_train_loader = build_dataloader(a_train_ds, anomaly_config["data"], "train", a_adapter,
                                   anomaly_ctx.seed, anomaly_config["runtime"]["device"])
a_valid_loader = build_dataloader(a_valid_ds, anomaly_config["data"], "valid", a_adapter,
                                   anomaly_ctx.seed, anomaly_config["runtime"]["device"])
a_optimizer = build_optimizer(anomaly_config["optim"], a_model)
a_scheduler = build_scheduler(anomaly_config["optim"], a_optimizer)

print("adapter.threshold before fit():", getattr(a_adapter, "threshold", None))

adapter.threshold before fit(): None


In [24]:
a_trainer = Trainer(checkpoint_dir=os.path.join(anomaly_run_dir, "checkpoints"))
a_trainer.fit(a_model, a_adapter, a_train_loader, a_valid_loader, a_optimizer, a_scheduler, anomaly_ctx)

print("adapter.threshold after fit():", a_adapter.threshold)

adapter.threshold after fit(): 27.16407012939453


`on_fit_start`는 반대 방향 사례를 보여준다 -- 학습을 시작하기 **전에** 모델이나 loader 전체를 봐야
하는 계산(예: 정상화 통계량 사전 계산)에 쓰인다. `src/tasks/anomaly/adapter.py`(실데이터
`AnomalyAdapter`)는 `model.on_fit_start`가 있으면 그것을 호출하도록 위임하는데, `EfficientAD`처럼
teacher 통계량을 학습 시작 전에 한 번 계산해야 하는 모델이 이 지점을 사용한다 (`tasks/04_anomaly.ipynb`
에서 다시 다룬다).

## 6. 엔진 순수성 -- 태스크 이름 분기가 없다

`CLAUDE.md`의 규칙은 "엔진(Trainer/Engine 루프)은 task-agnostic을 유지한다. 공통 루프에 태스크
이름으로 분기하는 조건문을 두지 않는다"다. `Trainer`의 전체 소스에서 태스크 이름 문자열을 검색해
실제로 없는지 확인한다.

In [25]:
import re

engine_source = inspect.getsource(Trainer)
task_names = ["cls", "seg", "det", "anomaly", "classification", "segmentation", "detection"]
# Word-boundary match: a naive substring check on "det" would also match "detach()" and
# "loss.detach" false positives that have nothing to do with the Detection task.
hits = {name: bool(re.search(rf"\b{name}\b", engine_source)) for name in task_names}
print(hits)
assert not any(hits.values()), "Trainer 소스에 태스크 이름 분기가 있으면 안 된다."
print("\nTrainer는 어떤 태스크 이름도 언급하지 않는다 -- 4장에서 본 대로, 태스크 차이는 전부")
print("adapter.train_step/eval_step/update_metrics/... 뒤로 숨어 있다.")

{'cls': False, 'seg': False, 'det': False, 'anomaly': False, 'classification': False, 'segmentation': False, 'detection': False}

Trainer는 어떤 태스크 이름도 언급하지 않는다 -- 4장에서 본 대로, 태스크 차이는 전부
adapter.train_step/eval_step/update_metrics/... 뒤로 숨어 있다.


## 요약

- Config는 `_base` 상속 + `--set` override로 만들어지고, 검증 실패는 `ConfigError`로 즉시 드러난다.
  `config.resolved.yaml`은 그 실행이 정확히 무엇으로 돌았는지 보여주는 사람이 읽기 편한 감사 자료다
  (체크포인트에도 같은 config가 저장되지만, 매번 `.pth`를 로드해 열어보는 대신 이 YAML 파일 하나만
  보면 된다는 뜻이다).
- Registry는 이름 -> 생성자 매핑일 뿐이다. 엔진은 이름으로 만든 객체가 무엇인지 몰라도 된다.
- `TaskAdapter`가 태스크 차이를 전부 흡수한다. `Trainer`는 7개 추상 메서드(`train_step`/
  `eval_step`/`update_metrics`/`compute_metrics`/`reset_metrics`/`predict_step`/`batch_size`)와
  4개 훅(`on_fit_start`/`on_fit_end`/`on_epoch_start`/`on_epoch_end`)만 호출한다.
- `Trainer.fit`은 `model.train()`/`model.eval()`/`no_grad()`/`reset_metrics()` 경계를 명확히
  지키고, monitor 기준으로만 best를 갱신한다. 순서는 best 체크포인트를 먼저 재로드 -> `on_fit_end`
  호출 -> 그 결과(버퍼 보정값 등)를 다시 `best.pth`에 저장, 이다 (반대 순서면 훅의 보정 결과가
  buffer에만 남고 저장되지 않거나, 훅이 아직 보정되지 않은 마지막 epoch 가중치를 보게 된다).
- `on_fit_start`/`on_fit_end`는 epoch 단위로 표현할 수 없는 계산(예: threshold 산출)을 위한
  탈출구이며, 기본은 no-op이다.
- `Trainer` 소스 어디에도 태스크 이름 분기가 없다 -- 코드로 검증했다.

다음 노트북(`01_toy_cls.ipynb` ~ `04_toy_anomaly.ipynb`)에서는 각 태스크의 target 규약이 이
계약 위에서 구체적으로 어떻게 흐르는지를 본다.